
## KOL Posts Data Cleaning Summary

This notebook cleans and prepares KOL post-level data to capture **content exposure and engagement**, aggregated at a restaurant–month level for post-clustering analysis. Output: Saved the cleaned and aggregated dataset for downstream merging and marketing effectiveness analysis after clustering

### What was done

* **Loaded raw KOL post data**

  * Imported post-level data containing views, likes, comments, and posting dates

* **Column standardisation**

  * Renamed key columns (e.g. `Restaurant Code` → `restaurant_id`, `Post Date` → `post_date`) for consistency with other datasets

* **Column filtering**

  * Retained only relevant fields required for engagement analysis (KOL identity, restaurant mapping, date, and engagement metrics)
  * Dropped captions, URLs, and other text-heavy fields not used in analysis

* **Data type cleaning**

  * Converted engagement metrics (`views`, `likes`, `comments`) from strings to numeric values
  * Treated placeholder values (e.g. `-1`) as missing data
  * Parsed `post_date` into datetime format using day–month–year convention
  * Cleaned and validated `restaurant_id` to ensure posts can be reliably linked to restaurants

* **Engagement feature engineering**

  * Created `engagement` by combining likes and comments to represent total user interaction
  * Created `engagement_per_view` to normalise engagement by reach, enabling fair comparison across posts with different view counts

* **Temporal alignment**

  * Derived a monthly time key `year_month` to align post performance with monthly momentum and clustering outputs

* **Aggregation**

  * Aggregated post-level data to a **restaurant–month** level, generating:

    * `kol_posts` — number of KOL posts in the month
    * `unique_kols` — number of distinct creators posting
    * `total_views` — total exposure from KOL posts
    * `avg_engagement_rate` — average engagement per view



1. load data

In [1]:
import pandas as pd
import numpy as np

posts = pd.read_csv("../data/kol/KOL_Posts.csv")


2. rename key columns 

Standardises column names so they match restaurant and momentum datasets, making merging easier later.

In [2]:
posts = posts.rename(columns={
    "Restaurant Code": "restaurant_id",
    "Post Date": "post_date",
    "Views": "views",
    "Likes": "likes",
    "Comments": "comments"
})


3. keep only valid/useful columns

In [3]:
POST_COLS = [
    "KOL_ID",
    "Username",
    "restaurant_id",
    "Cuisine Type",
    "Country",
    "post_date",
    "views",
    "likes",
    "comments"
]

posts = posts[POST_COLS]


4. clean numeric engagement columns

convert engagement metrics from string into usable numeric values

In [4]:
def clean_numeric(col):
    return (
        col.astype(str)
           .str.replace(",", "", regex=False)
           .replace("-1", np.nan)
           .astype(float)
    )

for c in ["views", "likes", "comments"]:
    posts[c] = clean_numeric(posts[c])


5. fix dates n restaurant IDs

Posts without a valid restaurant or date cannot be aligned with restaurant performance and are removed.

In [6]:
posts["post_date"] = pd.to_datetime(
    posts["post_date"],
    dayfirst=True,
    errors="coerce"
)

posts["restaurant_id"] = (
    pd.to_numeric(posts["restaurant_id"], errors="coerce")
    .astype("Int64")
)

posts = posts[posts["restaurant_id"].notna()]



6. create engagement features

Combines likes and comments into a single engagement metric to capture total user interaction with a post, then normalises this by views (engagement_per_view) so posts with different audience sizes can be compared fairly based on how engaging the content was, rather than how large the creator’s reach is.

In [7]:
posts["engagement"] = posts["likes"].fillna(0) + posts["comments"].fillna(0)
posts["engagement_per_view"] = posts["engagement"] / posts["views"]


7. create monthly time key

Converts post-level dates to month-level so they align with clustering and momentum outputs.

In [8]:
posts["year_month"] = (
    posts["post_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


8. Aggregate to restaurant–month

Combines all KOL posts for the same restaurant within a month into a single row, capturing how often the restaurant was promoted (kol_posts), how many different creators were involved (unique_kols), how much exposure it received (total_views), and how engaging the content was on average (avg_engagement_rate), so KOL performance can be directly compared with monthly restaurant performance and momentum results.

In [9]:
kol_posts_monthly = (
    posts
    .groupby(["restaurant_id", "year_month"], as_index=False)
    .agg(
        kol_posts=("KOL_ID", "count"),
        unique_kols=("KOL_ID", "nunique"),
        total_views=("views", "sum"),
        avg_engagement_rate=("engagement_per_view", "mean")
    )
)


In [11]:
kol_posts_monthly["restaurant_id"].nunique()


235

In [12]:
kol_posts_monthly["year_month"].nunique()


23

In [13]:
kol_posts_monthly.groupby("restaurant_id").size().describe()


count    235.000000
mean       1.821277
std        1.378240
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       11.000000
dtype: float64

9. save cleaned output

In [ ]:
kol_posts_monthly.to_parquet(
    "../data/kol/kol_cleaned_posts_monthly.parquet",
    index=False
)
